# DetectLLM reproduction (AI504 Assignment 2)

Reproduces LRR / NPR (Su et al., Findings of EMNLP 2023) and adds an Arabic extension.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
The full run takes about 2.5–3 hours. Keep this tab open. Results are saved to Google Drive, so if the session disconnects, run all cells again and finished runs are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Replace `YOUR_GITHUB_USERNAME` with your GitHub username after you have pushed the repository.
If the repository is not on GitHub yet, upload `detectllm-repro.zip` with the file panel on the left and set `USE_ZIP = True`.

In [ ]:
USE_ZIP = False
REPO = 'https://github.com/YOUR_GITHUB_USERNAME/detectllm-repro.git'
import os
os.chdir('/content')
if USE_ZIP:
    !unzip -q -o /content/detectllm-repro.zip -d /content
elif not os.path.exists('/content/detectllm-repro'):
    !git clone -q $REPO
os.chdir('/content/detectllm-repro')
!ls

## 2. Install and test

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q tests

## 3. Save results to Google Drive (recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/detectllm_results/full'
SMOKE = '/content/drive/MyDrive/detectllm_results/smoke'
os.environ['HF_HOME'] = '/content/hf_cache'

## 4. Smoke test (3–5 min)
This checks that downloads, generation, scoring and perturbation all work before the long run.

In [ ]:
!python scripts/run.py --preset smoke --out $SMOKE
!python scripts/aggregate.py --results $SMOKE
!cat $SMOKE/summary/E1_main.md | head -30

## 5. Full run (~2.5–3 h on a T4)
You can run experiment families separately, e.g. `--only E1`, then `--only E2`, and so on. Finished runs are skipped automatically.

In [ ]:
!python scripts/run.py --preset full --out $OUT 2>&1 | grep -v Warning

## 6. Tables, figures and error analysis

In [ ]:
!python scripts/aggregate.py --results $OUT
!ls $OUT/summary
!cat $OUT/summary/E1_claims.md
!cat $OUT/summary/E5_arabic.md

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(OUT + '/summary/fig_*.png')):
    print(f); display(Image(f, width=600))

## 7. Download the results
This zips the summary folder, the per-text scores, the timing log and the generated data, without the large caches. Send `results_full.zip` back so the report can be finalised.

In [ ]:
!cd $OUT/.. && zip -q -r /content/results_full.zip full/summary full/scores full/results.jsonl full/timing.jsonl full/data full/errors.log 2>/dev/null; ls -lh /content/results_full.zip
from google.colab import files
files.download('/content/results_full.zip')